# Data Preparation: Insufficient Physical Activity (WHO)

**Author:** Karin Pietruska  
**Date:** 15 November 2025


## 1. Purpose and dataset

This notebook prepares a reproducible analysis table of **age-standardized insufficient physical activity among adults aged 18+ years** from the World Health Organization Global Health Observatory (GHO).

| Item | Value |
|------|--------|
| Indicator | Prevalence of insufficient physical activity among adults aged 18+ years (age-standardized estimate) (%) |
| Indicator code | `NCD_PAA` |
| Source | [WHO GHO indicator page](https://www.who.int/data/gho/data/indicators/indicator-details/GHO/prevalence-of-insufficient-physical-activity-among-adults-aged-18-years-(age-standardized-estimate)-(-)) |
| API | [GHO OData API](https://www.who.int/data/gho/info/gho-odata-api) (`https://ghoapi.azureedge.net/api/NCD_PAA`) |

Age-standardization allows comparison across countries with different age structures. Each observation is a prevalence estimate (%) with a 95% confidence interval, disaggregated by geography, sex, and year.

**Default input:** a pinned snapshot at `data/raw/who_ncd_paa.json`. A normal **Run All** uses this file only and does not require internet access. Live API refresh is optional and does not overwrite the pinned file.

**Output:** `data/processed/who_physical_inactivity_clean.csv`, used by the exploratory analysis notebook.


## 2. Imports and configuration


In [5]:
from pathlib import Path
import json
import sys

import pandas as pd
import requests


In [6]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 0)

# Live WHO download is opt-in. Leave False for an offline, reproducible Run All.
REFRESH_FROM_API = False


## 3. Project paths

Paths are resolved from the repository root so the notebook works when the working directory is either `notebooks/` or the project root.


In [7]:
def find_project_root() -> Path:
    """Return the repository root containing `data/raw/who_ncd_paa.json`."""
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "data" / "raw" / "who_ncd_paa.json").exists():
            return candidate
        if (candidate / "environment.yml").exists() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root. Run this notebook from the repository "
        "(working directory: repo root or notebooks/)."
    )


PROJECT_ROOT = find_project_root()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

RAW_JSON = DATA_RAW / "who_ncd_paa.json"
COUNTRY_CSV = DATA_RAW / "who_ncd_paa_country.csv"
PROCESSED_CSV = DATA_PROCESSED / "who_physical_inactivity_clean.csv"
API_LATEST_JSON = DATA_RAW / "who_ncd_paa_latest.json"

print("Pinned raw JSON:", RAW_JSON.relative_to(PROJECT_ROOT))
print("Processed CSV:", PROCESSED_CSV.relative_to(PROJECT_ROOT))
assert RAW_JSON.exists(), f"Pinned snapshot not found: {RAW_JSON}"


Pinned raw JSON: data/raw/who_ncd_paa.json
Processed CSV: data/processed/who_physical_inactivity_clean.csv


## 4. Optional WHO API acquisition

The repository already contains a pinned GHO extract. That snapshot is the source of truth for this analysis.

The cell below is a **deliberate refresh path**. With `REFRESH_FROM_API = False` (the default), it does nothing and needs no network. Set the flag to `True` only if you want to download a newer extract.

A refresh writes `data/raw/who_ncd_paa_latest.json`. It does **not** overwrite `data/raw/who_ncd_paa.json`. To switch the analysis to a new extract, replace the pinned file manually after reviewing the download.

The indicator code `NCD_PAA` is already known. When refresh is enabled, the indicator catalog is queried only to confirm that code.


In [8]:
WHO_INDICATOR_API = "https://ghoapi.azureedge.net/api/Indicator"
WHO_NCD_PAA_API = "https://ghoapi.azureedge.net/api/NCD_PAA"

if not REFRESH_FROM_API:
    print("Skipping live WHO API requests (REFRESH_FROM_API=False).")
    print(f"Using pinned snapshot: {RAW_JSON.relative_to(PROJECT_ROOT)}")
else:
    print("REFRESH_FROM_API=True: downloading a fresh extract (pinned file will not be overwritten).")

    indicators_raw = pd.read_json(WHO_INDICATOR_API)
    indicators = pd.json_normalize(indicators_raw["value"])
    hits = indicators.loc[
        indicators["IndicatorName"].str.contains(
            "insufficient physical activity", case=False, na=False
        ),
        ["IndicatorCode", "IndicatorName"],
    ]
    display(hits)

    response = requests.get(WHO_NCD_PAA_API, timeout=120)
    response.raise_for_status()
    payload = response.json()
    records = payload["value"] if isinstance(payload, dict) and "value" in payload else payload

    API_LATEST_JSON.parent.mkdir(parents=True, exist_ok=True)
    with API_LATEST_JSON.open("w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(records):,} records to {API_LATEST_JSON.relative_to(PROJECT_ROOT)}")
    print(f"Pinned snapshot unchanged: {RAW_JSON.relative_to(PROJECT_ROOT)}")
    print("To use the new extract, replace who_ncd_paa.json with this file after review.")


Skipping live WHO API requests (REFRESH_FROM_API=False).
Using pinned snapshot: data/raw/who_ncd_paa.json


## 5. Load pinned raw dataset

The pinned JSON is the default input for all cleaning steps below.


In [9]:
with RAW_JSON.open("r", encoding="utf-8") as f:
    raw_records = json.load(f)

df = pd.DataFrame(raw_records)
print(f"Loaded {len(df):,} records from {RAW_JSON.name}")
df.head()


Loaded 14,214 records from who_ncd_paa.json


,Id,IndicatorCode,SpatialDimType,SpatialDim,TimeDimType,ParentLocationCode,ParentLocation,Dim1Type,TimeDim,Dim1,Dim2Type,Dim2,Dim3Type,Dim3,DataSourceDimType,DataSourceDim,Value,NumericValue,Low,High,Comments,Date,TimeDimensionValue,TimeDimensionBegin,TimeDimensionEnd
0,951007,NCD_PAA,REGION,WPR,YEAR,NaN,NaN,SEX,2010,SEX_MLE,AGEGROUP,AGEGROUP_YEARS18-PLUS,None,None,None,None,23.2 [19.8-26.8],23.225080,19.813980,26.825520,"In accordance with resolution WHA78.25 (2025),...",2025-10-30T14:12:02.71+01:00,2010,2010-01-01T00:00:00+01:00,2010-12-31T00:00:00+01:00
1,951833,NCD_PAA,COUNTRY,QAT,YEAR,EMR,Eastern Mediterranean,SEX,2010,SEX_MLE,AGEGROUP,AGEGROUP_YEARS18-PLUS,None,None,None,None,46.2 [33.5-58.9],46.199745,33.522572,58.878918,NaN,2024-06-21T13:42:48.827+02:00,2010,2010-01-01T00:00:00+01:00,2010-12-31T00:00:00+01:00
2,951955,NCD_PAA,COUNTRY,BRN,YEAR,WPR,Western Pacific,SEX,2019,SEX_FMLE,AGEGROUP,AGEGROUP_YEARS18-PLUS,None,None,None,None,36.2 [21.5-52.7],36.193363,21.513872,52.714764,NaN,2024-06-21T13:42:48.827+02:00,2019,2019-01-01T00:00:00+01:00,2019-12-31T00:00:00+01:00
3,952621,NCD_PAA,COUNTRY,BHR,YEAR,EMR,Eastern Mediterranean,SEX,2016,SEX_MLE,AGEGROUP,AGEGROUP_YEARS18-PLUS,None,None,None,None,34.3 [15.1-59.5],34.262978,15.071355,59.493984,NaN,2024-06-21T13:42:48.827+02:00,2016,2016-01-01T00:00:00+01:00,2016-12-31T00:00:00+01:00
4,952711,NCD_PAA,COUNTRY,MLI,YEAR,AFR,Africa,SEX,2010,SEX_BTSX,AGEGROUP,AGEGROUP_YEARS18-PLUS,None,None,None,None,33.9 [21.5-47.8],33.919476,21.537233,47.789452,NaN,2024-06-21T13:42:48.827+02:00,2010,2010-01-01T00:00:00+01:00,2010-12-31T00:00:00+01:00


### Optional country-name lookup file

`data/raw/who_ncd_paa_country.csv` is a GHO CSV export of **country-level** rows. It includes country names (`Location`) that the API extract does not. It is **not** the primary analysis source; cleaning uses the pinned JSON, which also includes region, global, and World Bank income-group rows.


In [10]:
if COUNTRY_CSV.exists():
    df_country_csv = pd.read_csv(COUNTRY_CSV)
    print(
        f"Reference CSV: {len(df_country_csv):,} rows, "
        f"{df_country_csv['SpatialDimValueCode'].nunique()} country codes, "
        f"columns include Location={('Location' in df_country_csv.columns)}"
    )
    display(df_country_csv[["SpatialDimValueCode", "Location", "ParentLocation", "Period", "Dim1"]].head())
else:
    print(f"Reference country CSV not found (optional): {COUNTRY_CSV.relative_to(PROJECT_ROOT)}")


Reference CSV: 13,455 rows, 195 country codes, columns include Location=True


,SpatialDimValueCode,Location,ParentLocation,Period,Dim1
0,RWA,Rwanda,Africa,2022,Female
1,NPL,Nepal,South-East Asia,2022,Male
2,COM,Comoros,Africa,2022,Male
3,NLD,Netherlands (Kingdom of the),Europe,2022,Female
4,TKM,Turkmenistan,Europe,2022,Male


## 6. Initial inspection

GHO records mix several geographic levels in `SpatialDimType`:

- **COUNTRY** — ISO 3166-1 alpha-3 codes (for example `DEU`)
- **REGION** — WHO regions (for example `EUR`, `AFR`)
- **GLOBAL** — world aggregate
- **WORLDBANKINCOMEGROUP** — World Bank income groups

Sex is stored in `Dim1` (`SEX_MLE`, `SEX_FMLE`, `SEX_BTSX`). Year is `TimeDimensionValue`. The estimate and 95% CI are `NumericValue`, `Low`, and `High`.


In [11]:
print("Shape:", df.shape)
print("\nColumn dtypes:")
print(df.dtypes)
print("\nGeographic level (SpatialDimType):")
print(df["SpatialDimType"].value_counts(dropna=False))
print("\nSex (Dim1):")
print(df["Dim1"].value_counts(dropna=False))
print("\nNumeric estimate summary:")
display(df[["NumericValue", "Low", "High"]].describe().round(2))
df.head()


Shape: (14214, 25)

Column dtypes:
Id                      int64
IndicatorCode             str
SpatialDimType            str
SpatialDim                str
TimeDimType               str
ParentLocationCode        str
ParentLocation            str
Dim1Type                  str
TimeDim                 int64
Dim1                      str
Dim2Type                  str
Dim2                      str
Dim3Type               object
Dim3                   object
DataSourceDimType      object
DataSourceDim          object
Value                     str
NumericValue          float64
Low                   float64
High                  float64
Comments                  str
Date                      str
TimeDimensionValue        str
TimeDimensionBegin        str
TimeDimensionEnd          str
dtype: object

Geographic level (SpatialDimType):
SpatialDimType
COUNTRY                 13455
REGION                    414
WORLDBANKINCOMEGROUP      276
GLOBAL                     69
Name: count, dtype: int64

Sex

,NumericValue,Low,High
count,14214.00,14214.00,14214.00
mean,26.52,16.60,38.80
std,11.48,10.09,14.68
min,2.39,0.72,5.25
25%,17.99,8.85,27.98
50%,24.59,14.46,37.06
75%,33.48,22.10,48.97
max,73.60,58.76,88.39


,Id,IndicatorCode,SpatialDimType,SpatialDim,TimeDimType,ParentLocationCode,ParentLocation,Dim1Type,TimeDim,Dim1,Dim2Type,Dim2,Dim3Type,Dim3,DataSourceDimType,DataSourceDim,Value,NumericValue,Low,High,Comments,Date,TimeDimensionValue,TimeDimensionBegin,TimeDimensionEnd
0,951007,NCD_PAA,REGION,WPR,YEAR,NaN,NaN,SEX,2010,SEX_MLE,AGEGROUP,AGEGROUP_YEARS18-PLUS,None,None,None,None,23.2 [19.8-26.8],23.225080,19.813980,26.825520,"In accordance with resolution WHA78.25 (2025),...",2025-10-30T14:12:02.71+01:00,2010,2010-01-01T00:00:00+01:00,2010-12-31T00:00:00+01:00
1,951833,NCD_PAA,COUNTRY,QAT,YEAR,EMR,Eastern Mediterranean,SEX,2010,SEX_MLE,AGEGROUP,AGEGROUP_YEARS18-PLUS,None,None,None,None,46.2 [33.5-58.9],46.199745,33.522572,58.878918,NaN,2024-06-21T13:42:48.827+02:00,2010,2010-01-01T00:00:00+01:00,2010-12-31T00:00:00+01:00
2,951955,NCD_PAA,COUNTRY,BRN,YEAR,WPR,Western Pacific,SEX,2019,SEX_FMLE,AGEGROUP,AGEGROUP_YEARS18-PLUS,None,None,None,None,36.2 [21.5-52.7],36.193363,21.513872,52.714764,NaN,2024-06-21T13:42:48.827+02:00,2019,2019-01-01T00:00:00+01:00,2019-12-31T00:00:00+01:00
3,952621,NCD_PAA,COUNTRY,BHR,YEAR,EMR,Eastern Mediterranean,SEX,2016,SEX_MLE,AGEGROUP,AGEGROUP_YEARS18-PLUS,None,None,None,None,34.3 [15.1-59.5],34.262978,15.071355,59.493984,NaN,2024-06-21T13:42:48.827+02:00,2016,2016-01-01T00:00:00+01:00,2016-12-31T00:00:00+01:00
4,952711,NCD_PAA,COUNTRY,MLI,YEAR,AFR,Africa,SEX,2010,SEX_BTSX,AGEGROUP,AGEGROUP_YEARS18-PLUS,None,None,None,None,33.9 [21.5-47.8],33.919476,21.537233,47.789452,NaN,2024-06-21T13:42:48.827+02:00,2010,2010-01-01T00:00:00+01:00,2010-12-31T00:00:00+01:00


## 7. Missing values

Key identifiers should be complete: geography (`SpatialDim`), sex (`Dim1`), and year (`TimeDimensionValue`). Empty GHO metadata fields (unused dimensions, data-source placeholders) are expected and are dropped later.


In [12]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).round(2)
missing_tbl = pd.DataFrame({"n_missing": missing, "pct": missing_pct})
display(missing_tbl[missing_tbl["n_missing"] > 0])

key_fields = ["SpatialDim", "Dim1", "TimeDimensionValue"]
print("Missing values in key fields:")
print(df[key_fields].isna().sum())


,n_missing,pct
Comments,14145,99.51
DataSourceDim,14214,100.00
DataSourceDimType,14214,100.00
Dim3,14214,100.00
Dim3Type,14214,100.00
ParentLocation,759,5.34
ParentLocationCode,759,5.34


Missing values in key fields:
SpatialDim            0
Dim1                  0
TimeDimensionValue    0
dtype: int64


## 8. Uniqueness of observations

An observation is uniquely identified by the composite key:

`SpatialDim` (place) + `Dim1` (sex) + `TimeDimensionValue` (year)

`SpatialDim` is a country, region, global, or income-group code depending on `SpatialDimType`. There should be no duplicate rows on this key.


In [13]:
n_full_dupes = int(df.duplicated().sum())
n_key_dupes = int(df.duplicated(subset=key_fields).sum())
print(f"Full-row duplicates: {n_full_dupes}")
print(f"Duplicates on {key_fields}: {n_key_dupes}")
if n_key_dupes:
    display(df.loc[df.duplicated(subset=key_fields, keep=False)].sort_values(key_fields))


Full-row duplicates: 0
Duplicates on ['SpatialDim', 'Dim1', 'TimeDimensionValue']: 0


## 9. Variable selection

Columns are dropped when they are empty, constant structural labels, redundant encodings of the same value, or metadata not used in this analysis.

| Reason | Columns |
|--------|---------|
| Empty / unused dimensions | `Dim3Type`, `Dim3`, `DataSourceDimType`, `DataSourceDim` |
| Constant structural labels | `IndicatorCode` (`NCD_PAA`), `Dim1Type` (`SEX`), `Dim2` / `Dim2Type` (adults 18+), `TimeDimType` (`YEAR`) |
| Redundant | `Value` (text form of the numeric estimate), `ParentLocationCode`, `TimeDim` (same year as `TimeDimensionValue`) |
| Metadata | `Date`, `TimeDimensionBegin`, `TimeDimensionEnd` |
| Documented then dropped | `Comments` |

`Comments` is non-empty only for the WHO note that, following resolution WHA78.25 (2025), Indonesia was reassigned to the Western Pacific Region and is included in those regional aggregates. That note is retained here in the documentation; the column is not needed as a data field.


In [14]:
print("Unique Comments values:")
print(df["Comments"].unique())

comment_rows = df["Comments"].str.contains("Western Pacific", case=False, na=False)
print(f"\nRows with the Indonesia / Western Pacific comment: {int(comment_rows.sum())}")
display(df.loc[comment_rows, ["SpatialDimType", "SpatialDim", "TimeDimensionValue"]].head(10))


Unique Comments values:
<StringArray>
['In accordance with resolution WHA78.25 (2025), Indonesia was reassigned to the WHO Western Pacific Region as of 27 May 2025. Data pertaining to Indonesia are therefore included in the Western Pacific regional aggregates.', nan]
Length: 2, dtype: str

Rows with the Indonesia / Western Pacific comment: 69


,SpatialDimType,SpatialDim,TimeDimensionValue
0,REGION,WPR,2010
15,REGION,WPR,2017
159,REGION,WPR,2014
461,REGION,WPR,2012
501,REGION,WPR,2014
577,REGION,WPR,2019
683,REGION,WPR,2021
1052,REGION,WPR,2001
1121,REGION,WPR,2018
1197,REGION,WPR,2019


In [15]:
drop_cols = [
    "Dim3Type", "DataSourceDimType", "DataSourceDim", "Dim3", "Comments",
    "IndicatorCode", "Dim2", "Dim2Type", "Dim1Type", "TimeDimType",
    "Value", "ParentLocationCode", "TimeDim", "Date", "TimeDimensionBegin", "TimeDimensionEnd",
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])
print(f"Remaining columns ({df.shape[1]}): {list(df.columns)}")
df.head()


Remaining columns (9): ['Id', 'SpatialDimType', 'SpatialDim', 'ParentLocation', 'Dim1', 'NumericValue', 'Low', 'High', 'TimeDimensionValue']


,Id,SpatialDimType,SpatialDim,ParentLocation,Dim1,NumericValue,Low,High,TimeDimensionValue
0,951007,REGION,WPR,NaN,SEX_MLE,23.225080,19.813980,26.825520,2010
1,951833,COUNTRY,QAT,Eastern Mediterranean,SEX_MLE,46.199745,33.522572,58.878918,2010
2,951955,COUNTRY,BRN,Western Pacific,SEX_FMLE,36.193363,21.513872,52.714764,2019
3,952621,COUNTRY,BHR,Eastern Mediterranean,SEX_MLE,34.262978,15.071355,59.493984,2016
4,952711,COUNTRY,MLI,Africa,SEX_BTSX,33.919476,21.537233,47.789452,2010


## 10. Renaming and recoding

WHO internal names are replaced with analysis names. The output schema is:

`who_id`, `spatial_type`, `country_code`, `region`, `sex`, `prevalence`, `ci_low`, `ci_high`, `year`

`country_code` keeps the GHO spatial code at **every** geographic level (ISO3 for countries; region/global/income-group codes otherwise). Always filter on `spatial_type` when selecting countries.


In [16]:
df = df.rename(columns={
    "Id": "who_id",
    "SpatialDimType": "spatial_type",
    "SpatialDim": "country_code",
    "ParentLocation": "region",
    "Dim1": "sex",
    "NumericValue": "prevalence",
    "Low": "ci_low",
    "High": "ci_high",
    "TimeDimensionValue": "year",
})

sex_mapping = {
    "SEX_MLE": "Male",
    "SEX_FMLE": "Female",
    "SEX_BTSX": "Both sexes",
}
spatial_mapping = {
    "COUNTRY": "Country",
    "REGION": "Region",
    "GLOBAL": "Global",
    "WORLDBANKINCOMEGROUP": "World Bank Income Group",
}

df["sex"] = df["sex"].replace(sex_mapping)
df["spatial_type"] = df["spatial_type"].replace(spatial_mapping)
df["year"] = pd.to_numeric(df["year"], errors="raise").astype(int)

column_order = [
    "who_id", "spatial_type", "country_code", "region",
    "sex", "prevalence", "ci_low", "ci_high", "year",
]
df = df[column_order]
df.head()


,who_id,spatial_type,country_code,region,sex,prevalence,ci_low,ci_high,year
0,951007,Region,WPR,NaN,Male,23.225080,19.813980,26.825520,2010
1,951833,Country,QAT,Eastern Mediterranean,Male,46.199745,33.522572,58.878918,2010
2,951955,Country,BRN,Western Pacific,Female,36.193363,21.513872,52.714764,2019
3,952621,Country,BHR,Eastern Mediterranean,Male,34.262978,15.071355,59.493984,2016
4,952711,Country,MLI,Africa,Both sexes,33.919476,21.537233,47.789452,2010


## 11. Final validation

Checks below describe this pinned snapshot (years 2000–2022, 195 countries). They will fail if a future extract changes coverage.


In [17]:
expected_columns = column_order
assert list(df.columns) == expected_columns, list(df.columns)

key = ["country_code", "sex", "year"]
n_key_dupes_clean = int(df.duplicated(subset=key).sum())
assert n_key_dupes_clean == 0, n_key_dupes_clean
assert df[key].notna().all().all()

for col in ["prevalence", "ci_low", "ci_high"]:
    df[col] = pd.to_numeric(df[col], errors="raise")

n_countries = df.loc[df["spatial_type"] == "Country", "country_code"].nunique()
year_min, year_max = int(df["year"].min()), int(df["year"].max())

print("spatial_type counts:")
print(df["spatial_type"].value_counts())
print(f"\nYear range: {year_min}–{year_max} ({df['year'].nunique()} distinct years)")
print(f"Country-level ISO3 codes: {n_countries}")
print(f"Rows: {len(df):,}")

assert year_min == 2000 and year_max == 2022, (year_min, year_max)
assert n_countries == 195, n_countries
assert df["prevalence"].between(0, 100).all()
assert (df["ci_low"] <= df["prevalence"]).all() and (df["prevalence"] <= df["ci_high"]).all()
print("\nValidation passed.")


spatial_type counts:
spatial_type
Country                    13455
Region                       414
World Bank Income Group      276
Global                        69
Name: count, dtype: int64

Year range: 2000–2022 (23 distinct years)
Country-level ISO3 codes: 195
Rows: 14,214

Validation passed.


## 12. Export processed dataset


In [18]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_CSV, index=False)

check = pd.read_csv(PROCESSED_CSV)
assert len(check) == len(df)
assert list(check.columns) == expected_columns
print(f"Wrote {len(check):,} rows to {PROCESSED_CSV.relative_to(PROJECT_ROOT)}")
check.head()


Wrote 14,214 rows to data/processed/who_physical_inactivity_clean.csv


,who_id,spatial_type,country_code,region,sex,prevalence,ci_low,ci_high,year
0,951007,Region,WPR,NaN,Male,23.225080,19.813980,26.825520,2010
1,951833,Country,QAT,Eastern Mediterranean,Male,46.199745,33.522572,58.878918,2010
2,951955,Country,BRN,Western Pacific,Female,36.193363,21.513872,52.714764,2019
3,952621,Country,BHR,Eastern Mediterranean,Male,34.262978,15.071355,59.493984,2016
4,952711,Country,MLI,Africa,Both sexes,33.919476,21.537233,47.789452,2010


## Appendix. WHO GHO field reference

Selected fields from the GHO OData record used in this notebook:

| Field | Role in this analysis |
|--------|------------------------|
| `IndicatorCode` | `NCD_PAA` (constant; dropped after confirmation) |
| `SpatialDimType` | Geographic level → `spatial_type` |
| `SpatialDim` | Place code → `country_code` |
| `ParentLocation` | WHO region name → `region` (empty for non-country rows) |
| `Dim1` | Sex → `sex` |
| `TimeDimensionValue` | Year → `year` |
| `NumericValue` | Age-standardized prevalence (%) → `prevalence` |
| `Low` / `High` | 95% confidence interval → `ci_low` / `ci_high` |
| `Id` | GHO record identifier → `who_id` |


## Environment

Library versions for this session. The full environment is declared in `environment.yml`.


In [19]:
print(f"Python {sys.version.split()[0]}")
print(f"pandas {pd.__version__}")
print(f"requests {requests.__version__}")


Python 3.11.15
pandas 3.0.5
requests 2.34.2
